# Ch 28 · Interop with PyTorch and JAX

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/28-interop-torch-and-jax.ipynb)

Companion notebook for the chapter page `chapters/28-interop-torch-and-jax.html` ([page](../chapters/28-interop-torch-and-jax.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## 0. JAX settings first
The setup cell above did not import JAX. The next cell puts the three XLA variables into the environment **before** anything imports JAX (Section 4 of the chapter explains each one).

In [ ]:
# JAX memory and graph settings: they must be in os.environ BEFORE the first `import jax`.
# XLA reads them once when its backend starts. If JAX was already imported in this runtime,
# use Runtime > Restart session and run this cell right after the setup cell.
import os
import sys
os.environ["XLA_FLAGS"] = "--xla_gpu_graph_min_graph_size=1"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
print("jax already imported:", "jax" in sys.modules)

In [ ]:
# Colab's preinstalled torch and JAX are used; never `pip install jax` here (it can replace the
# matched jax/jaxlib/CUDA plugin). Check what this runtime has:
import importlib.metadata as md
for dist in ("torch", "jax", "jaxlib"):
    try:
        print(dist, md.version(dist))
    except md.PackageNotFoundError:
        print(dist, "not installed")

# Part A: PyTorch

## 1. Zero-copy: one buffer, several views
`wp.to_torch(d.qpos)` is a torch tensor over MJWarp's own buffer. The example compares data pointers, writes from torch, steps with MJWarp and reads through the same view. A `d.qpos.numpy()` snapshot is a copy and goes stale.

In [ ]:
# Example 28.1 - ex1_torch_zero_copy.py
"""Example 28.1 - one buffer, two views: wp.to_torch(d.qpos) aliases MJWarp memory.

Prints the torch / Warp versions and devices, then shows that a torch view of d.qpos shares
the same memory (same data pointer), that a write from torch is seen by Warp and that a
Warp step is seen by torch, all without a copy. A d.qpos.numpy() snapshot, by contrast, is
a copy: it goes stale after the next step (on a GPU it is also a device-to-host transfer).
Runs on CPU torch; skips if torch is not installed.
Run: python ex1_torch_zero_copy.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed")

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 100

if torch is not None:
    # 1. Version block: Colab's torch is a CUDA 12 build; pip warp-lang 1.19.0 is a CUDA 13
    #    build. Both use the same driver, so the driver (R580+) is what must be new enough.
    print("torch", torch.__version__, "| torch CUDA build:", torch.version.cuda,
          "| torch.cuda.is_available():", torch.cuda.is_available())
    print("warp", wp.__version__, "| warp device:", wp.get_device(),
          "| as torch device:", wp.device_to_torch(wp.get_device()))

    mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

    # 2. A zero-copy view: no data moves, the tensor points at the Warp allocation.
    q = wp.to_torch(d.qpos)
    print("view:", tuple(q.shape), q.dtype, q.device, "| same memory:", q.data_ptr() == d.qpos.ptr)

    # 3. torch writes, Warp reads: put the ball at a different height in every world.
    q[:, 2] = torch.linspace(0.5, 2.0, NWORLD, device=q.device)
    print("z seen by Warp after the torch write:", d.qpos.numpy()[:4, 2])

    # 4. A host snapshot is a copy, the view is not.
    snapshot = d.qpos.numpy().copy()
    for _ in range(NSTEP):
        mjw.step(m, d)
    wp.synchronize()  # Warp work must finish before torch reads the memory (Example 28.2)
    print(f"after {NSTEP} steps, torch view z:", q[:4, 2].cpu().numpy())
    print("old numpy snapshot z (stale):    ", snapshot[:4, 2])
    print("view still aliases d.qpos:", q.data_ptr() == d.qpos.ptr)
    print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 2. Caveats: dtypes, shapes, contiguity, streams
Composite Warp types become trailing dims; strided slices are fine for scalar dtypes but not for vector dtypes; float64 tensors must be converted before `wp.copy` into MJWarp; part 5 (streams) runs only on a GPU.

In [ ]:
# Example 28.2 - ex2_torch_dtypes.py
"""Example 28.2 - dtypes, shapes, contiguity and streams at the Warp/torch boundary.

1. vec3 / quat / mat33 arrays become float32 tensors with trailing (3,) / (4,) / (3, 3) dims.
2. A non-contiguous torch slice: wp.from_torch keeps the strides for scalar dtypes (still a
   zero-copy view, is_contiguous False), but refuses vector dtypes whose inner dims are strided.
3. dtype mismatches: wp.from_torch maps float64 to wp.float64, and wp.copy into a float32
   MJWarp array then raises. wp.copy only compares element byte sizes, though: an int32 tensor
   (4 bytes, like float32) is copied bit for bit with no error. Convert in torch first.
4. uint32 buffers (the packed RGB of the batch renderer, Ch 27) come out as torch.uint32.
5. Streams (GPU only): Warp's default stream is a blocking stream, so it is ordered with torch's
   default stream; on a torch.cuda.Stream() (non-blocking) run Warp on that same stream.
Run: python ex2_torch_dtypes.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed")

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4


def show(name, t):
    print(f"  {name:<11} torch {str(tuple(t.shape)):<14} {t.dtype}  contiguous={t.is_contiguous()}")


if torch is not None:
    dev = wp.device_to_torch(wp.get_device())
    mjm = mujoco.MjModel.from_xml_string(PENDULUM)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qpos.assign(np.tile(mjm.key_qpos[0], (NWORLD, 1)).astype(np.float32))  # "horizontal"
    mjw.forward(m, d)

    # 1. composite dtypes gain trailing dims; worlds stay the leading dim
    print("1. Warp array -> torch view")
    for name in ("qpos", "xpos", "xquat", "xmat", "cvel"):
        arr = getattr(d, name)
        print(f"  d.{name:<9} warp {arr.dtype.__name__:<20} {arr.shape}")
    for name in ("xpos", "xquat", "xmat", "cvel"):
        show(name, wp.to_torch(getattr(d, name)))
    print("  link1 xquat (w, x, y, z), 90 deg about y:", wp.to_torch(d.xquat)[0, 1].cpu().numpy())

    # 2. contiguity: a column slice of a (nworld, 3) tensor is strided
    acts = torch.arange(NWORLD * 3, dtype=torch.float32, device=dev).reshape(NWORLD, 3)
    col = acts[:, 1]
    print("2. non-contiguous slice: stride", col.stride(), "contiguous", col.is_contiguous())
    w = wp.from_torch(col)
    print("  wp.from_torch(slice): shape", w.shape, "strides (bytes)", w.strides,
          "is_contiguous", w.is_contiguous)
    ctrl = wp.zeros((NWORLD,), dtype=float)
    wp.copy(ctrl, w)  # wp.copy handles the strided source
    print("  wp.copy(strided -> contiguous) ok:", ctrl.numpy()[:4])
    pairs = acts.reshape(3, NWORLD).t()  # (nworld, 3) view whose last dim is strided
    try:
        wp.from_torch(pairs, dtype=wp.vec3)
    except RuntimeError as e:
        print("  vec3 from strided inner dim -> RuntimeError:", str(e).split(", because ")[-1])
    print("  fix: wp.from_torch(pairs.contiguous(), dtype=wp.vec3).shape =",
          wp.from_torch(pairs.contiguous(), dtype=wp.vec3).shape)

    # 3. dtype: torch float64 maps to wp.float64; MJWarp arrays are float32
    a64 = torch.ones((NWORLD, mjm.nu), dtype=torch.float64, device=dev)
    print("3. torch.float64 -> warp", wp.from_torch(a64).dtype.__name__)
    try:
        wp.copy(d.ctrl, wp.from_torch(a64))
    except RuntimeError as e:
        print("  wp.copy(d.ctrl, float64) -> RuntimeError:", e)
    a32i = torch.arange(1, NWORLD + 1, dtype=torch.int32, device=dev).reshape(NWORLD, mjm.nu)
    wp.copy(d.ctrl, wp.from_torch(a32i))   # same element size: no error, bits reinterpreted
    print("  wp.copy(d.ctrl, int32 [1 2 3 4]) -> no error, d.ctrl =",
          " ".join(f"{x:.0e}" for x in d.ctrl.numpy()[:4, 0]), "(denormals)")
    wp.copy(d.ctrl, wp.from_torch(a64.float()))
    print("  after .float():", d.ctrl.numpy()[:, 0])

    # 4. uint32 buffers (packed RGB from the batch renderer) keep their dtype
    rgb_packed = wp.zeros((NWORLD, 8), dtype=wp.uint32)
    print("4. uint32 array -> torch", wp.to_torch(rgb_packed).dtype,
          "(few torch ops support it: unpack with mjw.get_rgb first, Ch 27)")

    # 5. streams: if torch works on its own non-blocking stream, put Warp on the same stream
    if ON_GPU:
        # stream_from_torch docstring: keep Warp-owned arrays used on a non-blocking stream
        # alive until that work is done (here d lives on; nothing temporary is launched).
        side = torch.cuda.Stream()
        torch_stream = wp.stream_from_torch(side)
        with torch.cuda.stream(side), wp.ScopedStream(torch_stream):
            mjw.step(m, d)                         # Warp launches go to torch's stream...
            tip_z = wp.to_torch(d.site_xpos)[:, 0, 2].mean()   # ...so torch reads after it
        torch.cuda.current_stream().wait_stream(side)
        print(f"5. stepped on torch's stream, mean tip z = {tip_z.item():.4f}")
    else:
        print("5. skipped on CPU: streams are CUDA only (wp.stream_from_torch)")

## 3. Feeding actions in
A torch policy on CARTPOLE: observations through views made once, actions copied **into** `d.ctrl` with `wp.from_torch` + `wp.copy`. The address of `d.ctrl` never changes.

In [ ]:
# Example 28.3 - ex3_torch_ctrl.py
"""Example 28.3 - a torch policy drives CARTPOLE: zero-copy obs out, actions copied in place.

Every step: torch reads qpos/qvel through zero-copy views, computes a linear feedback action
per world (gains scaled from 0 to 1 across the worlds), and the action tensor goes into d.ctrl
with wp.from_torch + wp.copy. d.ctrl is never replaced, so its address (and any view or CUDA
graph that holds it) stays valid. No training here: the gains come from an LQR design.
Run: python ex3_torch_ctrl.py [--dump-json PATH]
  --dump-json PATH writes decimated (cart x, pole angle) per world for the page's replay.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

try:
    import torch
except ImportError:
    torch = None
    print("skipped: torch not installed")

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 300
THETA0 = 0.2  # initial pole tilt [rad]; 0 = upright

if torch is not None:
    dev = wp.device_to_torch(wp.get_device())  # put torch work on Warp's device
    mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD)  # no contacts in this model

    d.qpos.assign(np.tile([0.0, THETA0], (NWORLD, 1)).astype(np.float32))
    ctrl_ptr = d.ctrl.ptr

    # zero-copy views, made once: they follow d.qpos / d.qvel forever (never replaced)
    q, v = wp.to_torch(d.qpos), wp.to_torch(d.qvel)
    K = torch.tensor([[0.9, 6.8, 1.1, 1.2]], device=dev)        # LQR gains (x, th, xd, thd)
    gain = torch.linspace(0.0, 1.0, NWORLD, device=dev)[:, None]  # world k: gain[k] * K

    frames = []
    for i in range(NSTEP):
        obs = torch.cat([q, v], dim=1)                     # (nworld, 4), reads Warp memory
        act = ((obs @ K.T) * gain).clamp(-1.0, 1.0)         # (nworld, 1) == d.ctrl.shape
        wp.copy(d.ctrl, wp.from_torch(act.contiguous()))   # copy INTO d.ctrl, same address
        mjw.step(m, d)
        if args.dump_json and i % 5 == 0:
            qp = d.qpos.numpy()
            frames.append([[round(float(a), 3) for a in row] for row in qp])
    wp.synchronize()

    theta = d.qpos.numpy()[:, 1]
    wrapped = (theta + np.pi) % (2 * np.pi) - np.pi  # a pole that fell may have spun around
    print("device:", wp.get_device(), "| nworld:", NWORLD, "| steps:", NSTEP)
    print("gain scale per world:", gain[:8, 0].cpu().numpy())
    print("final pole angle [rad]:", theta[:8])
    upright = np.abs(wrapped) < 0.05
    print(f"mean |wrapped pole angle| = {np.abs(wrapped).mean():.3f} rad; upright worlds: "
          f"{int(upright.sum())}/{NWORLD}")
    print("d.ctrl address unchanged:", d.ctrl.ptr == ctrl_ptr)

    if args.dump_json:
        with open(args.dump_json, "w") as f:
            json.dump({"nworld": NWORLD, "dt": 5 * float(mjm.opt.timestep), "kind": "cartpole",
                       "labels": [f"g={g:.2f}" for g in gain[:, 0].cpu().tolist()],
                       "frames": frames}, f, separators=(",", ":"))
        print("wrote", args.dump_json, len(frames), "frames")

## Exercises 28.1-28.3 (torch)
Shared setup for the three auto-checked exercises:

In [ ]:
import mujoco
import numpy as np
import torch
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

# CARTPOLE is the MJCF string defined in the Example 28.3 cell above
NWORLD = 8
mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
dev = wp.device_to_torch(wp.get_device())

### Exercise 28.1
Make a torch view of `d.ctrl` and write `k / 10` into world `k` with torch only.

In [ ]:
# Exercise 28.1: a write through the view
c = ...  # a torch view of d.ctrl
...      # write k / 10 into world k, with torch
assert np.allclose(d.ctrl.numpy()[:, 0], np.arange(NWORLD) / 10)
print("Exercise 28.1 passed")

In [ ]:
#@title Exercise 28.1: solution (double-click to show the code)
c = wp.to_torch(d.ctrl)
c[:, 0] = torch.arange(NWORLD, dtype=torch.float32, device=dev) / 10
assert np.allclose(d.ctrl.numpy()[:, 0], np.arange(NWORLD) / 10)
print("Exercise 28.1 passed")

### Exercise 28.2
Write `make_ctrl_writer(d)`: it returns `write(actions)` that copies a torch tensor of any float dtype into `d.ctrl` without replacing it.

In [ ]:
# Exercise 28.2: make_ctrl_writer(d) copies into d.ctrl without replacing it
def make_ctrl_writer(d):
    ...

before_obj, before_ptr = d.ctrl, d.ctrl.ptr
write = make_ctrl_writer(d)
write(torch.full((NWORLD, mjm.nu), 0.5, dtype=torch.float64, device=dev))
assert d.ctrl is before_obj and d.ctrl.ptr == before_ptr
assert np.allclose(d.ctrl.numpy(), 0.5)
mjw.step(m, d)
print("Exercise 28.2 passed")

In [ ]:
#@title Exercise 28.2: solution (double-click to show the code)
def make_ctrl_writer(d):
    target = d.ctrl                      # hold the array, not the attribute
    def write(actions):
        src = wp.from_torch(actions.to(torch.float32).contiguous())
        wp.copy(target, src)             # values move, the address stays
    return write

before_obj, before_ptr = d.ctrl, d.ctrl.ptr
write = make_ctrl_writer(d)
write(torch.full((NWORLD, mjm.nu), 0.5, dtype=torch.float64, device=dev))
assert d.ctrl is before_obj and d.ctrl.ptr == before_ptr
assert np.allclose(d.ctrl.numpy(), 0.5)
mjw.step(m, d)
print("Exercise 28.2 passed")

### Exercise 28.3
The actions are float64 **and** strided. Confirm the direct copy fails, then write `to_ctrl_tensor` so it works.

In [ ]:
# Exercise 28.3: sanitize awkward actions (float64 and strided)
raw = torch.linspace(-1, 1, 2 * NWORLD, dtype=torch.float64).reshape(NWORLD, 2)[:, :1]
assert not raw.is_contiguous() and raw.dtype == torch.float64
try:
    wp.copy(d.ctrl, wp.from_torch(raw))
    raise AssertionError("expected a RuntimeError")
except RuntimeError:
    pass

def to_ctrl_tensor(actions, d):
    ...

wp.copy(d.ctrl, wp.from_torch(to_ctrl_tensor(raw, d)))
assert np.allclose(d.ctrl.numpy(), raw.numpy())
print("Exercise 28.3 passed")

In [ ]:
#@title Exercise 28.3: solution (double-click to show the code)
raw = torch.linspace(-1, 1, 2 * NWORLD, dtype=torch.float64).reshape(NWORLD, 2)[:, :1]

def to_ctrl_tensor(actions, d):
    return actions.to(device=dev, dtype=torch.float32).reshape(d.ctrl.shape).contiguous()

wp.copy(d.ctrl, wp.from_torch(to_ctrl_tensor(raw, d)))
assert np.allclose(d.ctrl.numpy(), raw.numpy())
print("Exercise 28.3 passed")

# Part B: JAX
If JAX fails in this runtime, Part A is unaffected. Every JAX example checks that JAX and Warp are on the same kind of device; on a Colab GPU runtime they use `graph_mode=WARP`, on a CPU-only runtime `graph_mode=NONE`.

## 4. The XLA environment and a live DLPack view

In [ ]:
# Example 28.4 - ex4_jax_env_flags.py
"""Example 28.4 - the three XLA environment variables, set BEFORE the first `import jax`.

XLA reads these once, when its backend starts (the first jax computation or jax.devices() call);
later changes are ignored. Setting them before `import jax` is the one placement that is always
early enough:
  XLA_FLAGS=--xla_gpu_graph_min_graph_size=1  lets XLA capture one-op regions in its own CUDA
                                              graphs; for the Warp call that matters only in
                                              graph_mode=JAX (the Apollo notebook, which keeps
                                              the default); in WARP mode Warp captures the
                                              graph itself and the flag is harmless; set by
                                              jax_test.py, jax_unroll.py and Apollo
  XLA_PYTHON_CLIENT_ALLOCATOR=platform        allocate on demand, free when done (jax_test.py)
  XLA_PYTHON_CLIENT_PREALLOCATE=false         do not grab most of the GPU memory at start-up
Without the last two, JAX preallocates a large fraction of GPU memory and Warp (MJWarp's
buffers) can run out. Uses the installed JAX (on Colab: the preinstalled one); skips without it.
Then: wp.to_jax(d.qpos) is a zero-copy DLPack view, so the "immutable" jax.Array changes when
MJWarp steps. That is why the bridge in Example 28.5 copies in and out instead.
Run: python ex4_jax_env_flags.py
"""
import os
import sys

XLA_ENV = {
    "XLA_FLAGS": "--xla_gpu_graph_min_graph_size=1",
    "XLA_PYTHON_CLIENT_ALLOCATOR": "platform",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
}
already = "jax" in sys.modules  # True if something imported jax earlier in this process
for k, v in XLA_ENV.items():
    os.environ[k] = v
    print(f"{k}={os.environ[k]}")
if already:
    print("WARNING: jax was imported before these were set; restart the runtime to apply them")

try:
    import jax
except ImportError:
    jax = None
    print("skipped: jax not installed")

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

if jax is not None:
    print("jax", jax.__version__, "| default backend:", jax.default_backend(),
          "| devices:", jax.devices())
    print("warp", wp.__version__, "| warp device:", wp.get_device())
    same = (jax.default_backend() == "gpu") == wp.get_device().is_cuda
    print("JAX and Warp on the same kind of device:", same)
    if jax.default_backend() != "gpu":
        print("note: the GPU examples (WARP graph mode) need jax.default_backend() == 'gpu';"
              " Examples 28.5-28.6 fall back to the CPU path")

    mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=2, nconmax=4, njmax=16)
    jq = wp.to_jax(d.qpos)                       # DLPack: no copy
    z_before = np.asarray(jq[:, 2]).copy()
    for _ in range(100):
        mjw.step(m, d)
    wp.synchronize()
    print("wp.to_jax(d.qpos):", type(jq).__name__, jq.shape, jq.dtype)
    print("z read before steps:", z_before,
          "| same jax.Array after 100 steps:", np.asarray(jq[:, 2]))
    del jq  # release the DLPack view while Warp is still loaded (avoids a noisy exit)

## 4b. `mjw.step` as a JAX op: the corrected `jax_unroll`
A closure over `m, d` that copies in, steps and copies out, wrapped with `wp.jax_callable(..., graph_mode=..., output_dims=...)`, unrolled with `jax.lax.scan` inside `jax.jit`. The steps/s line is your own number.

In [ ]:
# Example 28.5 - ex5_jax_callable.py
"""Example 28.5 - mjw.step as a JAX op: the corrected course version of contrib/jax_unroll.py.

Fixes the three stale lines of contrib/jax_unroll.py (Example 28.7 finds them):
  * the model comes from the wheel, mujoco_warp/test_data/humanoid/humanoid.xml
    (the repo's '../benchmark' path is wrong, and benchmarks/ is not in the wheel);
  * nconmax is PER WORLD in mujoco-warp 4.0.1 (naconmax is the total), so 64, not 64 * NWORLD;
  * no wp.clear_kernel_cache() at import (it throws away every compiled kernel).
It follows _src/jax_test.py: XLA env vars before `import jax`, an explicit
m.opt.graph_conditional, a closure over m and d that copies in, steps and copies out, wrapped by
wp.jax_callable(..., graph_mode=wp.JaxCallableGraphMode.WARP, output_dims=...), unrolled with
jax.lax.scan inside jax.jit. NWORLD is fixed: output_dims bakes it into the op.
GPU (Colab): needs jax.default_backend() == "gpu" and a CUDA Warp device.
CPU fallback: Warp 1.19 also runs jax_callable on the JAX CPU backend with graph_mode NONE
(WARP graph modes need CUDA). Skips if JAX is not installed or the two disagree on the device.
Run: python ex5_jax_callable.py
"""
import os

os.environ["XLA_FLAGS"] = "--xla_gpu_graph_min_graph_size=1"   # all three BEFORE import jax
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

try:
    import jax
    from jax import numpy as jp
except ImportError:
    jax = None
    print("skipped: jax not installed")

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
UNROLL_LENGTH = 1000 if ON_GPU else 20

if jax is not None:
    JAX_GPU = jax.default_backend() == "gpu"
    if JAX_GPU != ON_GPU:
        print(f"skipped: JAX backend {jax.default_backend()} but Warp device {wp.get_device()}")
        jax = None

if jax is not None:
    GRAPH_MODE = wp.JaxCallableGraphMode.WARP if ON_GPU else wp.JaxCallableGraphMode.NONE

    path = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"  # fix 1
    mjm = mujoco.MjModel.from_xml_path(path.as_posix())
    mjm.opt.iterations = 1
    mjm.opt.ls_iterations = 4
    mjd = mujoco.MjData(mjm)
    mjd.qvel = np.random.default_rng(0).uniform(-0.01, 0.01, mjm.nv)  # a small kick
    mujoco.mj_step(mjm, mjd, 3)
    mujoco.mj_forward(mjm, mjd)

    m = mjw.put_model(mjm)
    # explicit, as in jax_test.py: conditional graph nodes only with CUDA >= 12.4 (Ch 24)
    m.opt.graph_conditional = ON_GPU and wp.is_conditional_graph_supported()
    d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=64, njmax=128)  # fix 2: per world

    def warp_step(
        qpos_in: wp.array2d[wp.float32],
        qvel_in: wp.array2d[wp.float32],
        qpos_out: wp.array2d[wp.float32],
        qvel_out: wp.array2d[wp.float32],
    ):
        wp.copy(d.qpos, qpos_in)    # copy IN: JAX owns qpos_in, d keeps its own buffers
        wp.copy(d.qvel, qvel_in)
        mjw.step(m, d)
        wp.copy(qpos_out, d.qpos)   # copy OUT into the buffers JAX allocated
        wp.copy(qvel_out, d.qvel)

    warp_step_fn = wp.jax_callable(
        warp_step,
        num_outputs=2,
        graph_mode=GRAPH_MODE,
        output_dims={"qpos_out": (NWORLD, mjm.nq), "qvel_out": (NWORLD, mjm.nv)},
    )

    def unroll(qpos, qvel):
        def step(carry, _):
            qpos, qvel = carry
            qpos, qvel = warp_step_fn(qpos, qvel)  # returns a list: unpack it
            return (qpos, qvel), None

        (qpos, qvel), _ = jax.lax.scan(step, (qpos, qvel), length=UNROLL_LENGTH)
        return qpos, qvel

    jax_qpos = jp.tile(jp.array(d.qpos.numpy()[0]), (NWORLD, 1))
    jax_qvel = jp.tile(jp.array(d.qvel.numpy()[0]), (NWORLD, 1))

    jax_unroll_fn = jax.jit(unroll).lower(jax_qpos, jax_qvel).compile()
    jax.block_until_ready(jax_unroll_fn(jax_qpos, jax_qvel))  # warm up (JIT, graph capture)

    beg = time.perf_counter()
    final_qpos, final_qvel = jax_unroll_fn(jax_qpos, jax_qvel)
    jax.block_until_ready((final_qpos, final_qvel))
    run_time = time.perf_counter() - beg

    qpos = np.asarray(final_qpos)
    print("jax", jax.__version__, "| backend:", jax.default_backend(), "| warp device:",
          wp.get_device(), "| graph_mode:", GRAPH_MODE.name,
          "| graph_conditional:", m.opt.graph_conditional)
    print("nworld:", NWORLD, "| unroll:", UNROLL_LENGTH, "| naconmax:", d.naconmax,
          "(= nconmax 64 x nworld)")
    print("final qpos:", qpos.shape, type(final_qpos).__name__, "| all finite:",
          bool(np.isfinite(qpos).all()))
    print("root height z per world:", qpos[:4, 2])
    print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())
    label = "GPU" if ON_GPU else "CPU dev box, not a GPU number"
    print(f"steps per second ({label}): {NWORLD * UNROLL_LENGTH / run_time:,.0f}")

## 5. A compiled control loop with rewards in JAX
Policy and reward in `jax.numpy`, the simulator as one op in the scan. The last line shows that `jax.grad` through the op raises (Ch 29).

In [ ]:
# Example 28.6 - ex6_jax_grad_free_loop.py
"""Example 28.6 - a jax.jit-compiled control loop around the MJWarp op, rewards computed in JAX.

CARTPOLE with a linear policy written in JAX: every scan step reads the state that the
jax_callable returned, computes ctrl in JAX, calls the Warp op (ctrl, qpos, qvel in -> qpos,
qvel out) and adds a reward. The whole rollout is one compiled XLA program. There are no
gradients: jax.grad through the Warp op raises, because the FFI call has no JVP/VJP (and a
custom_vjp would need backward rules that MJWarp 4.0.1 does not provide, Ch 29). The last lines
print that error.
GPU: graph_mode WARP. CPU fallback: graph_mode NONE (see Example 28.5).
Run: python ex6_jax_grad_free_loop.py [--dump-json PATH]
  --dump-json PATH writes the per-step reward of every world (decimated) for the page's chart.
"""
import os

os.environ["XLA_FLAGS"] = "--xla_gpu_graph_min_graph_size=1"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

try:
    import jax
    from jax import numpy as jp
except ImportError:
    jax = None
    print("skipped: jax not installed")

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 1000 if ON_GPU else 200

if jax is not None and (jax.default_backend() == "gpu") != ON_GPU:
    print(f"skipped: JAX backend {jax.default_backend()} but Warp device {wp.get_device()}")
    jax = None

if jax is not None:
    mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
    m = mjw.put_model(mjm)
    m.opt.graph_conditional = ON_GPU and wp.is_conditional_graph_supported()
    d = mjw.make_data(mjm, nworld=NWORLD)

    def warp_step(
        ctrl_in: wp.array2d[wp.float32],
        qpos_in: wp.array2d[wp.float32],
        qvel_in: wp.array2d[wp.float32],
        qpos_out: wp.array2d[wp.float32],
        qvel_out: wp.array2d[wp.float32],
    ):
        wp.copy(d.ctrl, ctrl_in)
        wp.copy(d.qpos, qpos_in)
        wp.copy(d.qvel, qvel_in)
        mjw.step(m, d)
        wp.copy(qpos_out, d.qpos)
        wp.copy(qvel_out, d.qvel)

    mode = wp.JaxCallableGraphMode.WARP if ON_GPU else wp.JaxCallableGraphMode.NONE
    step_fn = wp.jax_callable(
        warp_step, num_outputs=2, graph_mode=mode,
        output_dims={"qpos_out": (NWORLD, mjm.nq), "qvel_out": (NWORLD, mjm.nv)})

    K = jp.array([0.9, 6.8, 1.1, 1.2])          # LQR gains, as in Example 28.3
    gain = jp.linspace(0.0, 1.0, NWORLD)          # world k uses gain[k] * K

    def rollout(qpos, qvel, gain):
        def body(carry, _):
            qpos, qvel, ret = carry
            obs = jp.concatenate([qpos, qvel], axis=1)              # (nworld, 4)
            ctrl = jp.clip((obs @ K) * gain, -1.0, 1.0)[:, None]    # policy in JAX
            qpos, qvel = step_fn(ctrl, qpos, qvel)                 # MJWarp step
            x, th = qpos[:, 0], qpos[:, 1]
            reward = jp.cos(th) - 0.1 * x**2 - 0.001 * ctrl[:, 0] ** 2  # reward in JAX
            return (qpos, qvel, ret + reward), reward   # carry, per-step output

        (qpos, qvel, ret), rewards = jax.lax.scan(
            body, (qpos, qvel, jp.zeros(NWORLD)), length=NSTEP)
        return ret / NSTEP, rewards                   # rewards: (NSTEP, NWORLD)

    qpos0 = jp.tile(jp.array([0.0, 0.2]), (NWORLD, 1))   # pole tilted 0.2 rad
    qvel0 = jp.zeros((NWORLD, mjm.nv))
    fn = jax.jit(rollout)
    avg_reward, rewards = jax.block_until_ready(fn(qpos0, qvel0, gain))

    r = np.asarray(avg_reward)
    print("backend:", jax.default_backend(), "| graph_mode:", mode.name, "| nworld:", NWORLD,
          "| steps:", NSTEP)
    print("gain scale:     ", np.asarray(gain)[:8])
    print("average reward: ", r[:8])
    print(f"best world: {int(r.argmax())} (gain {float(gain[r.argmax()]):.3f}), "
          f"reward {r.max():.3f}; worst: {r.min():.3f}")

    try:  # no gradients through the Warp op
        jax.grad(lambda g: fn(qpos0, qvel0, g)[0].sum())(gain)
    except Exception as e:
        print("jax.grad ->", type(e).__name__ + ":", str(e).splitlines()[0])

    if args.dump_json:
        rw = np.asarray(rewards)[::4, :8]  # every 4th step, first 8 worlds
        with open(args.dump_json, "w") as f:
            json.dump({"dt": 4 * float(mjm.opt.timestep),
                       "gain": np.asarray(gain)[:8].round(3).tolist(),
                       "avg": r[:8].round(3).tolist(), "frames": rw.round(3).tolist()},
                      f, separators=(",", ":"))
        print("wrote", args.dump_json, len(rw), "frames")

## 6. What is stale in `contrib/jax_unroll.py`
The next cell clones the repository (optional; about 1 minute). Example 28.7 reads the script as text and prints the stale lines with their fixes. Without the clone it prints a skip line and then only the wheel checks.

In [ ]:
# Optional: a clone of the library repository for Example 28.7 (read as text, never executed).
# Cloned to mjw_src/ so the folder name cannot shadow the installed mujoco_warp package.
import os
!git clone -q https://github.com/google-deepmind/mujoco_warp.git mjw_src && git -C mjw_src checkout -q 8f5b9f1
os.environ["MUJOCO_WARP_SRC"] = "mjw_src"

In [ ]:
# Example 28.7 - ex7_stale_contrib_check.py
"""Example 28.7 - a static check of contrib/jax_unroll.py: find the stale lines, print the fixes.

Reads contrib/jax_unroll.py as TEXT from a clone of the mujoco_warp repository; nothing from the
clone is imported or executed. For each stale pattern it prints the line number, the line, and
the corrected course line (Example 28.5). It also checks that the claims hold for the installed
4.0.1 wheel. Skips if no clone is found.
Clone (optional): git clone https://github.com/google-deepmind/mujoco_warp.git
Run: python ex7_stale_contrib_check.py [--src PATH_TO_CLONE]   (or set MUJOCO_WARP_SRC)
"""
import argparse
import os
import pathlib
import re

import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--src", default=os.environ.get("MUJOCO_WARP_SRC"))
args, _ = parser.parse_known_args()

candidates = [args.src] if args.src else ["mujoco_warp", "../mujoco_warp", "/content/mujoco_warp"]
script = next((pathlib.Path(c) / "contrib/jax_unroll.py" for c in candidates
               if (pathlib.Path(c) / "contrib/jax_unroll.py").is_file()), None)

CHECKS = [  # (what is stale, regex on the line, corrected course line)
    ("model path: the repo folder is 'benchmarks' and it is not in the wheel",
     r"\.\./benchmark\b",
     'path = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"'),
    ("nconmax is per world in 4.0.1 (naconmax is the total)",
     r"nconmax\s*=\s*64\s*\*\s*NWORLDS",
     "d = mjw.put_data(mjm, mjd, nworld=NWORLD, nconmax=64, njmax=128)"),
    ("clears the whole Warp kernel cache at import: every run recompiles",
     r"wp\.clear_kernel_cache\(\)",
     "(deleted)"),
    ("also fragile: hard-coded 8192 instead of NWORLDS; qpos0/zero qvel drop the kick",
     r"\(8192\b",
     "jax_qpos = jp.tile(jp.array(d.qpos.numpy()[0]), (NWORLD, 1)); same for jax_qvel"),
]

if script is None:
    print("skipped: no mujoco_warp clone found (pass --src or set MUJOCO_WARP_SRC)")
else:
    lines = script.read_text().splitlines()
    print(f"checking {script} ({len(lines)} lines)")
    for what, pattern, fix in CHECKS:
        hits = [(i + 1, ln) for i, ln in enumerate(lines) if re.search(pattern, ln)]
        if not hits:
            print(f"\n[not found] {what}  (already fixed upstream?)")
            continue
        for no, ln in hits:
            print(f"\n[{what}]\n  line {no:>2}: {ln.strip()}\n  course:  {fix}")
    # Order check: XLA reads its environment once; set it before `import jax` (Example 28.4).
    first = lambda pat: next((i + 1 for i, ln in enumerate(lines) if re.search(pat, ln)), None)
    jax_line, flags_line = first(r"^\s*import jax\b"), first(r"environ\[.XLA_FLAGS.\]")
    missing = [v for v in ("XLA_PYTHON_CLIENT_ALLOCATOR", "XLA_PYTHON_CLIENT_PREALLOCATE")
               if not any(v in ln for ln in lines)]
    if jax_line and flags_line and flags_line > jax_line:
        print(f"\n[also fragile: XLA_FLAGS set after import jax; works only because the backend starts later]"
              f"\n  line {jax_line:>2}: import jax\n  line {flags_line:>2}: {lines[flags_line - 1].strip()}"
              f"\n  course:  all three XLA variables in os.environ before import jax")
    if missing:
        print("  never set:", ", ".join(missing), "(JAX may preallocate most of the GPU memory)")

# The claims, checked against the installed wheel (no clone needed):
pkg = pathlib.Path(str(epath.resource_path("mujoco_warp")))
print("\nwheel has test_data/humanoid/humanoid.xml:",
      (pkg / "test_data/humanoid/humanoid.xml").is_file(),
      "| wheel has ../benchmark/humanoid:", (pkg.parent / "benchmark/humanoid").exists())
doc = mjw.put_data.__doc__
print("put_data docstring:", " ".join(re.search(r"nconmax: (.*?)\.", doc).group(1).split()))
NWORLDS = 8192
print(f"nconmax=64*NWORLDS with NWORLDS={NWORLDS} asks for naconmax = "
      f"{64 * NWORLDS * NWORLDS:,} contacts (intended: {64 * NWORLDS:,})")

## 7. The `custom_vmap` pattern
`jax.vmap` over a per-env step becomes one call of the batched Warp op. The counter shows how often the Python Warp function ran (on a GPU in WARP mode it counts captures). The eager loop stays at 8 envs even on a GPU: in WARP mode the graph-cache key includes a `call_id`, and every eager Python call of a `jax_callable` gets a new one, so each eager 1-world call captures a new CUDA graph. A jitted function traces once and keeps one `call_id`.

In [ ]:
# Example 28.8 - ex8_custom_vmap.py
"""Example 28.8 - the custom_vmap pattern (Apollo notebook): many per-env calls, one Warp call.

RL libraries such as Brax write the env step for ONE environment and jax.vmap it over the batch.
A Warp op with fixed output_dims cannot be vmapped like a jnp function, so the Apollo notebook
(contrib/apptronik_apollo_locomotion.ipynb) wraps the per-env step in
jax.custom_batching.custom_vmap: the vmap rule receives the whole batch and calls the batched
MJWarp op once. This example counts how often the Python Warp function runs:
  * loop: NLOOP (at most 8) separate calls of a 1-world op;   * vmap: one call of the NWORLD-world op.
wp.jax_callable has vmap_method="broadcast_all", but that folds the batch into each array's first
dim, which a function closing over a fixed-size mjw.Data cannot accept; hence custom_vmap.
The loop is capped: on a GPU in WARP mode the graph-cache key is (device, call_id, buffer
addresses), and every eager Python call gets a new call_id, so each one misses the cache and
captures a new CUDA graph of mjw.step. A jitted function traces once and keeps one call_id.
Runs on JAX CPU (graph_mode NONE) or GPU (WARP); skips without JAX.
Run: python ex8_custom_vmap.py
"""
import os

os.environ["XLA_FLAGS"] = "--xla_gpu_graph_min_graph_size=1"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

try:
    import jax
    from jax import numpy as jp
except ImportError:
    jax = None
    print("skipped: jax not installed")

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 8192 if ON_GPU else 8
NLOOP = min(NWORLD, 8)  # per-env eager calls to compare (each gets a new call_id: a new capture on GPU)

if jax is not None and (jax.default_backend() == "gpu") != ON_GPU:
    print(f"skipped: JAX backend {jax.default_backend()} but Warp device {wp.get_device()}")
    jax = None

if jax is not None:
    mjm = mujoco.MjModel.from_xml_string(PENDULUM)
    m = mjw.put_model(mjm)
    m.opt.graph_conditional = ON_GPU and wp.is_conditional_graph_supported()
    mode = wp.JaxCallableGraphMode.WARP if ON_GPU else wp.JaxCallableGraphMode.NONE
    calls = {"n": 0}

    def make_op(nworld):
        """A jax op stepping `nworld` worlds: (ctrl, qpos, qvel) -> (qpos, qvel)."""
        d = mjw.make_data(mjm, nworld=nworld)

        def warp_step(ctrl_in: wp.array2d[wp.float32], qpos_in: wp.array2d[wp.float32],
                      qvel_in: wp.array2d[wp.float32], qpos_out: wp.array2d[wp.float32],
                      qvel_out: wp.array2d[wp.float32]):
            calls["n"] += 1  # Python side: runs once per FFI execution without graph replay
            wp.copy(d.ctrl, ctrl_in)
            wp.copy(d.qpos, qpos_in)
            wp.copy(d.qvel, qvel_in)
            mjw.step(m, d)
            wp.copy(qpos_out, d.qpos)
            wp.copy(qvel_out, d.qvel)

        return wp.jax_callable(warp_step, num_outputs=2, graph_mode=mode,
                               output_dims={"qpos_out": (nworld, mjm.nq),
                                            "qvel_out": (nworld, mjm.nv)})

    op1, opN = make_op(1), make_op(NWORLD)

    # The per-env step an RL library wants to vmap: shapes (nu,), (nq,), (nv,).
    @jax.custom_batching.custom_vmap
    def env_step(ctrl, qpos, qvel):
        qpos, qvel = op1(ctrl[None], qpos[None], qvel[None])  # unbatched call: 1 world
        return qpos[0], qvel[0]

    @env_step.def_vmap
    def env_step_vmap(axis_size, in_batched, ctrl, qpos, qvel):
        assert axis_size == NWORLD, "the Warp op is built for a fixed NWORLD"
        ctrl, qpos, qvel = (x if b else jp.tile(x, (axis_size, 1))  # tile unbatched inputs
                            for x, b in zip((ctrl, qpos, qvel), in_batched))
        qpos, qvel = opN(ctrl, qpos, qvel)                     # ONE batched MJWarp call
        return (qpos, qvel), (True, True)                      # both outputs are batched

    ctrl = jp.linspace(-1.0, 1.0, NWORLD)[:, None]            # a different torque per env
    qpos = jp.tile(jp.array(mjm.key_qpos[0]), (NWORLD, 1))
    qvel = jp.zeros((NWORLD, mjm.nv))

    calls["n"] = 0
    loop = [env_step(ctrl[i], qpos[i], qvel[i]) for i in range(NLOOP)]
    jax.block_until_ready(loop)
    n_loop = calls["n"]
    q_loop = np.stack([np.asarray(q) for q, _ in loop])

    calls["n"] = 0
    q_vmap, _ = jax.block_until_ready(jax.jit(jax.vmap(env_step))(ctrl, qpos, qvel))
    n_vmap = calls["n"]

    print("backend:", jax.default_backend(), "| graph_mode:", mode.name, "| envs:", NWORLD)
    print(f"python loop over the first {NLOOP} envs: {n_loop} Warp calls of a 1-world op")
    print(f"jax.vmap + custom_vmap: {n_vmap} Warp call of the {NWORLD}-world op")
    print("hinge1 after one step, loop:", q_loop[:4, 0])
    print("hinge1 after one step, vmap:", np.asarray(q_vmap)[:4, 0])
    print("max |loop - vmap| over those envs:",
          float(np.abs(q_loop - np.asarray(q_vmap)[:NLOOP]).max()))

## Stretch: `mjw.forward` as a JAX op
Wrap `mjw.forward` so it maps `(qpos, qvel)` to `qacc`, wrap `mjw.step` too, and check `qvel + h * qacc == qvel_step` on the double pendulum (Euler, no damping). Then: what do the two ops share through one `d`? What does `jax.grad` do?

In [ ]:
# your code here

In [ ]:
#@title Stretch: solution (double-click to show the code)
# Stretch: mjw.forward as a JAX op, checked against the step op (double pendulum, Euler).
import jax
from jax import numpy as jp

PEND_XML = """
<mujoco model="double_pendulum">
  <option timestep="0.01"/>
  <worldbody>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      </body>
    </body>
  </worldbody>
</mujoco>
"""
ON_GPU = wp.get_device().is_cuda
NW = 8
mode = wp.JaxCallableGraphMode.WARP if ON_GPU else wp.JaxCallableGraphMode.NONE
pm = mujoco.MjModel.from_xml_string(PEND_XML)
pw = mjw.put_model(pm)
pw.opt.graph_conditional = ON_GPU and wp.is_conditional_graph_supported()
pd = mjw.make_data(pm, nworld=NW)

def warp_forward(qpos_in: wp.array2d[wp.float32], qvel_in: wp.array2d[wp.float32],
                 qacc_out: wp.array2d[wp.float32]):
    wp.copy(pd.qpos, qpos_in); wp.copy(pd.qvel, qvel_in)
    mjw.forward(pw, pd)
    wp.copy(qacc_out, pd.qacc)

def warp_step(qpos_in: wp.array2d[wp.float32], qvel_in: wp.array2d[wp.float32],
              qpos_out: wp.array2d[wp.float32], qvel_out: wp.array2d[wp.float32]):
    wp.copy(pd.qpos, qpos_in); wp.copy(pd.qvel, qvel_in)
    mjw.step(pw, pd)
    wp.copy(qpos_out, pd.qpos); wp.copy(qvel_out, pd.qvel)

fwd = wp.jax_callable(warp_forward, num_outputs=1, graph_mode=mode,
                      output_dims={"qacc_out": (NW, pm.nv)})
stp = wp.jax_callable(warp_step, num_outputs=2, graph_mode=mode,
                      output_dims={"qpos_out": (NW, pm.nq), "qvel_out": (NW, pm.nv)})
qpos = jp.stack([jp.linspace(0.2, 1.5, NW), jp.zeros(NW)], axis=1)
qvel = jp.zeros((NW, pm.nv))
(qacc,) = fwd(qpos, qvel)
qpos1, qvel1 = stp(qpos, qvel)
err = float(jp.abs(qvel + pm.opt.timestep * qacc - qvel1).max())
print("max |qvel + h*qacc - qvel_step| =", err)
assert err < 1e-5